# Writing client code: notebooks and apps

This notebook is for whoever writes code against `ess.dispatch`: a notebook, a local app, a service-backed app, or a program.
It follows one thread, from a for loop over runs to a sum of runs that grows as they arrive.
Every step shows what the framework guarantees, and what the client must do in return.

The workflows are the toy SANS reduction of `toy_sans.py`.
A run holds counts per pixel and a monitor count; `IOFQ` reduces one sample run and one can run to an I(Q) curve, `IOFQ_MULTI` sums several of each, and `FORWARD` reads a curve and returns its first point.
[workflow-authors.ipynb](workflow-authors.ipynb) shows how these specs and their bindings are written.
Here they are given, as an installed package would give them.

| Spec | Params | Outputs |
|---|---|---|
| `IOFQ` | `sample: Run`, `can: Run`, `reduction: Reduction` (`bins=2`) | `iofq` |
| `IOFQ_MULTI` | `sample_runs: list[Run]`, `can_runs: list[Run]`, `reduction: Reduction` | `iofq` |
| `FORWARD` | `curve: Curve` (`data`) | `i0` |

In [1]:
from ess.dispatch import BLANK, Backend, Client, Request, SubmitError, Template, dataset
from ess.dispatch.testing import FakeDatasets, FakeStore
from toy_sans import (
    BINDINGS,
    CAN_RUNS,
    SAMPLE_RUNS,
    FORWARD,
    IOFQ,
    IOFQ_MULTI,
    Curve,
    ForwardParams,
    IofQParams,
    MultiIofQParams,
    Reduction,
    Run,
)

## Setup

The framework reads raw data through a *dataset source*.
Here a fake one holds a few runs of proposal `p1`, and a fake store stands in for disk.

In [2]:
datasets = FakeDatasets(proposal='p1')
for run, data in SAMPLE_RUNS.items():
    datasets.measure(run, data, role='sample', temperature=f'{run + 149}K')
for run, data in CAN_RUNS.items():
    datasets.measure(run, data, role='can')

backend = Backend(datasets, BINDINGS, store=FakeStore())
client = Client(backend, proposal='p1', submitter='alice')

`local(proposal='p1', datasets=datasets, bind=BINDINGS, store=...)` does the same in one call.
This notebook keeps `backend` to open a second client later, as another process would with `connect(url, proposal='p1')` on the service.
The service is not implemented; everything below runs in this process.

## 1. From a for loop to records

A notebook today calls the reduction directly:

```python
curves = {}
for run in (101, 102, 103):
    curves[run] = reduce_iofq(sample=run, can=110, bins=2)
```

With the framework, each call becomes a request: a spec and its parameter values.
`submit` returns a *record* at once, and the requests run in parallel.

In [3]:
can = Run(file=dataset(run=110))
records = {}
for run in (101, 102, 103):
    params = IofQParams(sample=Run(file=dataset(run=run)), can=can)
    records[run] = client.submit(IOFQ, params)
client.wait(records)
client.output(records[101], 'iofq')

<scipp.Variable> (Q: 2)    float64  [dimensionless]  [1.2, 0.4]

A record holds the request as the framework accepted it: every value filled in, defaults included, and each dataset resolved from the name the user gave (`run:101`) to its identity.
It holds no output values.
A record never changes; only its status moves from `pending` to `completed`, `failed`, or `cancelled`.

In [4]:
record = records[101]
record.request.params, client.status(record)

({'sample': {'file': DatasetRef(dataset='uuid:run-101')},
  'can': {'file': DatasetRef(dataset='uuid:run-110')},
  'reduction': {'bins': 2}},
 <Status.COMPLETED: 'completed'>)

Every call takes one record, a list, or a dict, and returns the same shape:

In [5]:
client.status(records)

{101: <Status.COMPLETED: 'completed'>,
 102: <Status.COMPLETED: 'completed'>,
 103: <Status.COMPLETED: 'completed'>}

**A request that cannot run is refused at submission**, with an error that names the field.
If one request of a call is refused, none is submitted, so a batch with one typo starts nothing.

In [6]:
batch = {
    'good': {'sample': {'file': dataset(run=102)}, 'can': can},
    'typo': {'sample': {'file': dataset(run=102)}, 'can': can, 'reduction': {'bins': 0}},
}
try:
    client.submit({key: Request(IOFQ, params) for key, params in batch.items()})
except SubmitError as error:
    print(error)

typo: toy-sans-iofq/v1: reduction.bins: Input should be greater than 0


Values may be given as models or as dicts of their fields; both make the same request.
`Request(spec, params)` makes a request without submitting it, so that several go into one call.

## 2. Chains: a reference to a pending record

An input that holds data is given by *reference*: to a dataset, or to an output of a record.
The record may still be pending, so a chain is submitted without waiting.
The framework runs `FORWARD` once the I(Q) has completed.

In [7]:
curve = client.submit(IOFQ, IofQParams(sample=Run(file=dataset(run=103)), can=can))
i0 = client.submit(FORWARD, ForwardParams(curve=Curve(data=curve.ref('iofq'))))
client.output(i0, 'i0')

<scipp.Variable> ()    float64  [dimensionless]  1

`client.output` waits for the record.
The framework checks at submission that the output fits the field: both declare the same `ArraySpec`, dimension `Q` and unit dimensionless.

## 3. Who keeps a value

An output value, such as the I(Q) array, lives while something keeps it, and nothing else keeps it:

| Keeper | Keeps the value until |
|---|---|
| the client that submitted the record | the client releases the record, or ends |
| a pending request that reads it | that request's workflow has returned |
| a stage, an accumulator's template, or a push that references it | the binding has taken it, or the stage, accumulator, or push is released or stopped |
| a persist request | the value is written |
| the store, once the value is written | the proposal's history is dropped |

A record object, a reference, or a label keeps nothing.
Releasing drops the client's hold; the record stays, but its value is gone for good:

In [8]:
client.release(curve)
try:
    client.output(curve, 'iofq')
except LookupError as error:
    print(error)

record 437ed691e55f46c1814483b42abc3d5c is not kept by this client


`FORWARD` had read it already, so `i0` is unaffected.
Had `FORWARD` still been pending, it would have kept the curve until it ran.

Two consequences that a client author must know:

- **Work that nothing keeps is cancelled.** Releasing a pending record that no pending request reads cancels it, and closing a client releases everything it keeps. A notebook that submits a batch and closes its client before the batch has run cancels the batch, unless it was persisted.
- **Another client never reads the values this client keeps**, not even in the same process. It reads only persisted values.

On the service, the memory a client keeps is capped, and a client that makes no call for a while is ended (its *lease*).
A long-lived app therefore releases what it no longer shows.

## 4. Persist, and find it the next day

`persist=` at submission makes the store the only keeper.
This is how an unattended batch is submitted: the client keeps nothing, and the records complete once their outputs are written.
A *label* names the batch, and a dict's keys become the *members*:

In [9]:
night = client.submit(
    {
        f'{run + 149}K': Request(IOFQ, IofQParams(sample=Run(file=dataset(run=run)), can=can))
        for run in (101, 102, 103)
    },
    label='scan',
    persist=True,
)
client.wait(night)

{'250K': <Status.COMPLETED: 'completed'>,
 '251K': <Status.COMPLETED: 'completed'>,
 '252K': <Status.COMPLETED: 'completed'>}

The next morning, another client of the same proposal finds the batch by its label and reads from the store.
Here a second client in this process stands in for it:

In [10]:
morning = Client(backend, proposal='p1', submitter='bob')
latest = morning.latest('scan', member='250K')
morning.output(latest, 'iofq')

<scipp.Variable> (Q: 2)    float64  [dimensionless]  [1.2, 0.4]

`client.persist(record)` instead adds the store next to the client, for a record the client keeps.
On the service, records are the proposal's history: they last until the proposal has been idle for days to weeks, and are then dropped as a whole.
With `local()`, history is held in memory and ends with the process.
What must last longer is published to SciCat (not implemented).

## 5. Templates: tune with a stage

A *template* is a spec with some parameters fixed and others left blank.
A *stage* is a template the backend keeps for this client, so that what depends only on the fixed values is computed once.
Tuning `bins` then does not load the runs again:

In [11]:
tune = client.stage(
    Template.from_spec(
        IOFQ, sample=Run(file=dataset(run=101)), can=can, reduction=BLANK
    )
)
for bins in (1, 2, 4):
    tuned = client.compute(tune, {'reduction': Reduction(bins=bins)})
    print(bins, client.output(tuned, 'iofq').values)

1 [1.6]
2 [1.2 0.4]
4 [0.7 0.5 0.3 0.1]


A call through a stage makes the record of the plain request, the one with every value given.
Nothing in the record says it went through a stage, and its outputs are those of the plain request (the *caching* symmetry):

In [12]:
tuned.request.params

{'sample': {'file': DatasetRef(dataset='uuid:run-101')},
 'can': {'file': DatasetRef(dataset='uuid:run-110')},
 'reduction': {'bins': 4}}

How much a stage saves depends on the binding. A binding built from a sciline pipeline computes the fixed part once; a plain function computes everything at every call.

## 6. Sum runs as they arrive: an accumulator

An *accumulator* is a template whose blanks are tables, kept by the backend for this client.
Each *push* adds a row to one or more tables.
State n, the accumulator after n pushes, has the outputs of the plain request whose tables hold the rows of the first n pushes.

In [13]:
total = client.accumulator(
    Template.from_spec(IOFQ_MULTI, sample_runs=BLANK, can_runs=BLANK)
)
total.push({'sample_runs': Run(file=dataset(run=101)), 'can_runs': can})  # state 1
state_1 = client.output(total, 'iofq')
total.push({'sample_runs': Run(file=dataset(run=102))})  # state 2
state_2 = client.output(total, 'iofq')
state_1.values, state_2.values

(array([1.2, 0.4]), array([0.8       , 0.13333333]))

`push` returns once the backend has logged it; the backend adds it later.
A read *pins* the state after the pushes logged so far and waits for it.

State 2 equals the plain request over the same rows (the *arrival* symmetry):

In [14]:
plain = client.compute(
    IOFQ_MULTI,
    MultiIofQParams(
        sample_runs=[Run(file=dataset(run=101)), Run(file=dataset(run=102))],
        can_runs=[can],
    ),
)
client.output(plain, 'iofq').values

array([0.8       , 0.13333333])

The binding of `IOFQ_MULTI` accumulates: its *held state*, which the backend keeps for the accumulator, holds the sums of numerators and denominators, and a push loads one run and adds to them.
With a binding that does not, the held state keeps the rows, and the backend reduces all of them for each state that is read.
The outputs are the same; the cost is not, and the client cannot tell which it gets.

**Readers never see a value change.**
`client.output` of an accumulator returns a copy, so `state_1` above still holds state 1.
A request may read a state too, through `total.ref(...)`.
It reads the output of the held state without a copy, so the backend holds back the next push until that request's workflow has returned:

In [15]:
forward_2 = client.submit(FORWARD, ForwardParams(curve=Curve(data=total.ref('iofq'))))  # pins state 2
total.push({'sample_runs': Run(file=dataset(run=103)), 'can_runs': Run(file=dataset(run=111))})
client.output(forward_2, 'i0').value, state_2['Q', 0].value

(np.float64(0.8), np.float64(0.8))

The order of the calls is the order of the states, as if each call ran after the one before it, although pushes and reads run in parallel.

A row repeated in a table is refused, since pushing a run twice is usually a cell run twice:

In [16]:
try:
    total.push({'sample_runs': Run(file=dataset(run=103))})
except SubmitError as error:
    print(error)

sample_runs[3]: repeats sample_runs[2]


At the end, `freeze` turns the last state into an ordinary record, without a copy, and the accumulator takes no more pushes:

In [17]:
summed = client.freeze(total)
client.output(summed, 'iofq').values

array([0.85, 0.25])

## 7. Where a result came from

Every record can say which requests and datasets produced it.
The record of a frozen state names the plain request over its rows, not the accumulator:

In [18]:
provenance = client.provenance(summed)
[str(d) for d in provenance.datasets()]

['uuid:run-101',
 'uuid:run-102',
 'uuid:run-103',
 'uuid:run-110',
 'uuid:run-111']

## 8. Flavours of client code

Everything above is the same API for every client.
What differs between flavours is which calls they rely on, and what they must take care of.

| Flavour | Makes its client with | Typically | Must take care of |
|---|---|---|---|
| notebook | `local(...)` | `submit`, `output`, stages, accumulators | releasing large values; not modifying values that `output` returned (see below) |
| local app (not a notebook) | `local(...)` | the same, behind a form | releasing what it no longer shows; pairing runs into rows |
| app against the service | `connect(url, proposal=...)` (not implemented) | `submit(..., persist=True)`, labels, `latest` | the memory cap and the lease; reading results from the store |
| batch or automatic reduction | either | `apply`, `Rule`, `TriggerLoop` | choosing what to persist |
| program, CLI, agent | `connect(...)` (not implemented) | `SpecId`, `SerializedWorkflowSpec` | importing no workflow package |

Points that hold for apps but not for notebooks:

- **Forms come from the spec.** `IOFQ.serialize()` gives the JSON Schema of the params. An app that imports only a package's specs, kept in a module apart from the workflow code, builds forms without the science stack ([ADR 0007](../adr/0007-packages-split-by-dependencies.md)).
- **Errors name the field at fault**, in the message of a `SubmitError` (`typo: toy-sans-iofq/v1: reduction.bins: ...` above). The error carries no structured field path, so a form that shows the error next to its field must read it from the message.
- **An app that does not import the workflow package** names the spec by `SpecId(name='toy-sans-iofq', version=1)` and gets the same checks.
- **The framework knows no technique.** Which sample run goes with which can run, and which runs form a row, is the app's decision.

**Batch reduction** fills a template once per dataset with `apply`, which submits nothing and returns the requests keyed by a metadata field:

In [19]:
from ess.dispatch import Selector, apply

samples = client.datasets.list(Selector(role='sample'))
requests = apply(
    Template.from_spec(IOFQ, sample=BLANK, can=can),
    samples,
    client.datasets,
    member_field='temperature',
)
batch = client.submit(requests, label='batch', persist=True)
client.wait(batch)
sorted(r.member for r in client.records(label='batch'))

['250K', '251K', '252K']

A `Rule` holds the same as plain data (a template, a selector, a label), and the `TriggerLoop` submits it for each new dataset; see [automatic-reduction.md](../automatic-reduction.md).

**Values in the user's process are not copied.**
In a notebook, `client.output` of a record returns the value the framework holds, not a copy.
Modifying it in place changes what later requests read and what `persist` writes.
This is open question 7 of the [README](../README.md); until it is settled, treat outputs as read-only.

## Summary: what client code relies on, and what it owes

| Relies on | Owes |
|---|---|
| a record never changes, only its status | keeping (or persisting) what it still needs, and releasing the rest |
| a refused call submits nothing, and names the field | treating output values as read-only |
| a value lives while something keeps it, and only then | pairing runs into rows: the framework knows no technique |
| a request runs once the records it references have completed | |
| a reader never sees a value change | |
| a call through a stage, or a state of an accumulator, gives the outputs of its plain request | |
| where a request runs changes neither its record nor its outputs | |

[framework-promises.ipynb](framework-promises.ipynb) shows what keeping these promises takes, in this process and on the service.

In [20]:
client.close()
morning.close()
backend.close()